# Fashion MNIST Pro: End-to-End MLOps Pipeline
**Custom ResNet Architecture & Explainable AI (XAI)**

This notebook represents the final Inference & Evaluation stage. We evaluate our custom Residual Network (ResNet) and use Grad-CAM to understand its decision-making process.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
print(f"✅ Environment initialized. TF Version: {tf.__version__}")

In [ ]:
class ResidualBlock(layers.Layer):
    def __init__(self, filters, stride=1, **kwargs):
        super(ResidualBlock, self).__init__(**kwargs)
        self.stride = stride
        self.conv1 = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)
        self.bn1 = layers.BatchNormalization()
        self.conv2 = layers.Conv2D(filters, 3, strides=1, padding="same", use_bias=False)
        self.bn2 = layers.BatchNormalization()
        if self.stride != 1:
            self.shortcut_conv = layers.Conv2D(filters, 1, strides=stride, use_bias=False)
            self.shortcut_bn = layers.BatchNormalization()

    def call(self, inputs):
        x = layers.ReLU()(self.bn1(self.conv1(inputs)))
        x_processed = self.bn2(self.conv2(x))
        if self.stride != 1:
            shortcut = self.shortcut_conv(inputs)
            shortcut = self.shortcut_bn(shortcut)
        else:
            shortcut = inputs
        x = layers.add([x_processed, shortcut])
        return layers.ReLU()(x)

print("Loading production model...")
model = tf.keras.models.load_model('models/resnet_model.keras', custom_objects={'ResidualBlock': ResidualBlock})
model.summary()

## Performance Evaluation & Explainable AI

Below is our **Confusion Matrix** (showing where the model excels and struggles) and our **Grad-CAM Heatmap** (showing which pixels the model focused on to make its decision).

![Confusion Matrix](outputs/confusion_matrix.png)

![Grad-CAM](outputs/gradcam_heatmap.png)